PLSR

In [39]:
import numpy as np, pandas as pd
from scipy.signal import savgol_filter
from sklearn.model_selection import RepeatedKFold
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score, mean_squared_error

CSV = "SpectroFood_dataset.csv"
SPLIT_CSV = "plsr_split_ks70_15_15.csv"
PREPROC = "snv_sg1"
SEED = 42
MAX_LV = 10
CV = RepeatedKFold(n_splits=10, n_repeats=3, random_state=SEED)

df = pd.read_csv(CSV)
ID_COL, TARGET_COL = "Mush No", "Dry matter"
ids = df[ID_COL].values
y = df[TARGET_COL].values.astype(float)
X = df.drop(columns=[ID_COL, TARGET_COL]).values.astype(float)

def snv(A): return (A - A.mean(1, keepdims=True)) / A.std(1, keepdims=True)
def preprocess(A, m):
    if m == "raw": return A
    if m == "snv": return snv(A)
    if m == "snv_sg1": return savgol_filter(snv(A), 11, 2, deriv=1, axis=1)
    raise ValueError(m)

def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def metrics(t, p):
    r = rmse(t, p)
    return dict(R2=r2_score(t, p), RMSE=r, RPD=np.std(t, ddof=1) / r,
                RPIQ=(np.percentile(t, 75) - np.percentile(t, 25)) / r,
                Bias=float(np.mean(p - t)))

Xp = preprocess(X, PREPROC)

sp = pd.read_csv(SPLIT_CSV).set_index("Mush No").loc[ids].reset_index()
split = sp["split"].values
cal = np.where(split == "cal")[0]
val = np.where(split == "val")[0]
tst = np.where(split == "test")[0]
Xc, yc = Xp[cal], y[cal]
Xv, yv = Xp[val], y[val]
Xt, yt = Xp[tst], y[tst]
print(f"Cal {len(cal)} | Val {len(val)} | Test {len(tst)}")

# ---- LV selection: repeated CV on calibration only ----
rmsecv = []
for lv in range(1, MAX_LV + 1):
    pred, cnt = np.zeros_like(yc), np.zeros_like(yc)
    for tr, va in CV.split(Xc):
        m = PLSRegression(n_components=lv, scale=False).fit(Xc[tr], yc[tr])
        pred[va] += m.predict(Xc[va]).ravel(); cnt[va] += 1
    rmsecv.append(rmse(yc, pred / cnt))
best_lv = int(np.argmin(rmsecv)) + 1
print("Best LV:", best_lv, "| RMSECV:", round(min(rmsecv), 5))

# RMSECV vs LV curve (use this instead of an "epoch" curve)
pd.DataFrame({"LV": range(1, MAX_LV + 1), "RMSECV": rmsecv}).to_csv("plsr_rmsecv_curve.csv", index=False)

# ---- Final model ----
pls = PLSRegression(n_components=best_lv, scale=False).fit(Xc, yc)
pc, pv, pt = (pls.predict(A).ravel() for A in (Xc, Xv, Xt))

out = {"Model": "PLSR", "Preproc": PREPROC, "Params": f"LV={best_lv}",
       "Split": f"loaded from {SPLIT_CSV}",
       "RMSECV": min(rmsecv), "RMSEV": rmse(yv, pv)}
out.update({f"{k}_cal":  v for k, v in metrics(yc, pc).items()})
out.update({f"{k}_val":  v for k, v in metrics(yv, pv).items()})
out.update({f"{k}_test": v for k, v in metrics(yt, pt).items()})
res = pd.DataFrame([out]).round(4)
print(res.T)

res.to_csv("plsr_results_ks70_15_15.csv", index=False)
pd.DataFrame({"Mush No": ids[cal], "y_true": yc, "y_pred": pc}).to_csv("plsr_cal_pred_ks70_15_15.csv", index=False)
pd.DataFrame({"Mush No": ids[val], "y_true": yv, "y_pred": pv}).to_csv("plsr_val_pred_ks70_15_15.csv", index=False)
pd.DataFrame({"Mush No": ids[tst], "y_true": yt, "y_pred": pt}).to_csv("plsr_test_pred_ks70_15_15.csv", index=False)

Cal 175 | Val 38 | Test 37
Best LV: 10 | RMSECV: 0.14815
                                               0
Model                                       PLSR
Preproc                                  snv_sg1
Params                                     LV=10
Split      loaded from plsr_split_ks70_15_15.csv
RMSECV                                    0.1481
RMSEV                                      0.122
R2_cal                                    0.8077
RMSE_cal                                   0.127
RPD_cal                                    2.287
RPIQ_cal                                  1.6635
Bias_cal                                    -0.0
R2_val                                    0.7905
RMSE_val                                   0.122
RPD_val                                   2.2139
RPIQ_val                                  1.6616
Bias_val                                 -0.0077
R2_test                                   0.7861
RMSE_test                                 0.1475
RPD_test    

In [24]:
import warnings
import numpy as np, pandas as pd
from scipy.signal import savgol_filter
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore", message="Some inputs do not have OOB scores")

CSV = "SpectroFood_dataset.csv"
SPLIT_CSV = "plsr_split_ks70_15_15.csv"   # <-- SAME split as PLSR
PREPROC = "snv_sg1"
SEED    = 42
EPOCHS  = 100          # one tree added per epoch

df = pd.read_csv(CSV)
ids, y = df["Mush No"].values, df["Dry matter"].values.astype(float)
X = df.iloc[:, 2:].values.astype(float)

def snv(A): return (A - A.mean(1, keepdims=True)) / A.std(1, keepdims=True)
def preprocess(A, m):
    if m == "raw": return A
    if m == "snv": return snv(A)
    if m == "snv_sg1": return savgol_filter(snv(A), 11, 2, deriv=1, axis=1)
    raise ValueError(m)

def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def metrics(t, p):
    r = rmse(t, p)
    return dict(R2=r2_score(t, p), RMSE=r, RPD=np.std(t, ddof=1) / r,
                RPIQ=(np.percentile(t, 75) - np.percentile(t, 25)) / r,
                Bias=float(np.mean(p - t)))

Xp = preprocess(X, PREPROC)

# ---- Load the SAME split from PLSR ----
sp = pd.read_csv(SPLIT_CSV).set_index("Mush No").loc[ids].reset_index()
split = sp["split"].values
cal = np.where(split == "cal")[0]
val = np.where(split == "val")[0]
tst = np.where(split == "test")[0]
Xc, yc = Xp[cal], y[cal]
Xv, yv = Xp[val], y[val]
Xt, yt = Xp[tst], y[tst]
print(f"Cal {len(cal)} | Val {len(val)} | Test {len(tst)} | EPOCHS={EPOCHS}")

# ---- Hyperparameter tuning (5-fold CV on calibration only) ----
# Regularized grid: shallow trees, bigger leaves, fewer features/split, subsampling, pruning
grid = {"max_depth": [3, 5, 8],
        "min_samples_leaf": [3, 5, 10],
        "max_features": [0.05, 0.1, 0.2],
        "max_samples": [0.5, 0.7, 1.0],
        "ccp_alpha": [0.0, 1e-4]}
gs = GridSearchCV(RandomForestRegressor(n_estimators=EPOCHS, bootstrap=True,
                                        random_state=SEED, n_jobs=-1),
                  grid, cv=KFold(5, shuffle=True, random_state=SEED),
                  scoring="neg_root_mean_squared_error", n_jobs=1).fit(Xc, yc)
print("Best params:", gs.best_params_, "| RMSECV:", round(-gs.best_score_, 5))
bp = gs.best_params_

# ---- Train with 100 epochs: 1 tree added per epoch via warm_start ----
rf = RandomForestRegressor(
    n_estimators=1,
    warm_start=True,
    random_state=SEED,
    max_depth=bp["max_depth"],
    min_samples_leaf=bp["min_samples_leaf"],
    max_features=bp["max_features"],
    max_samples=bp["max_samples"],
    ccp_alpha=bp["ccp_alpha"],
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
)

history = []
for epoch in range(1, EPOCHS + 1):
    rf.n_estimators = epoch           # add one tree this epoch
    rf.fit(Xc, yc)
    history.append((epoch,
                    rmse(yc, rf.predict(Xc)),
                    rmse(yv, rf.predict(Xv)),
                    rf.oob_score_))
hist_df = pd.DataFrame(history, columns=["epoch", "RMSEC", "RMSEV", "OOB_R2"])
hist_df.to_csv("rf_history_ks70_15_15.csv", index=False)
print(hist_df.tail(5).to_string(index=False))

# ---- Final predictions ----
pc = rf.predict(Xc)
pv = rf.predict(Xv)
pt = rf.predict(Xt)
rmse_oob = rmse(yc, rf.oob_prediction_)     # honest calibration error for RF

out = {"Model": "RF", "Preproc": PREPROC,
       "Params": f"{bp}, epochs={EPOCHS}",
       "Split": f"loaded from {SPLIT_CSV}",
       "RMSECV": -gs.best_score_, "RMSE_oob": rmse_oob, "RMSEV": rmse(yv, pv)}
out.update({f"{k}_cal":  v for k, v in metrics(yc, pc).items()})
out.update({f"{k}_val":  v for k, v in metrics(yv, pv).items()})
out.update({f"{k}_test": v for k, v in metrics(yt, pt).items()})
res = pd.DataFrame([out]).round(4)
print(res.T)

res.to_csv("rf_results_ks70_15_15.csv", index=False)
pd.DataFrame({"Mush No": ids[cal], "y_true": yc, "y_pred": pc}).to_csv("rf_cal_pred_ks70_15_15.csv", index=False)
pd.DataFrame({"Mush No": ids[val], "y_true": yv, "y_pred": pv}).to_csv("rf_val_pred_ks70_15_15.csv", index=False)
pd.DataFrame({"Mush No": ids[tst], "y_true": yt, "y_pred": pt}).to_csv("rf_test_pred_ks70_15_15.csv", index=False)

wl = df.columns[2:].astype(float).values
pd.DataFrame({"wavelength": wl, "importance": rf.feature_importances_}).to_csv("rf_importance_ks70_15_15.csv", index=False)

Cal 175 | Val 38 | Test 37 | EPOCHS=100
Best params: {'ccp_alpha': 0.0, 'max_depth': 8, 'max_features': 0.2, 'max_samples': 0.7, 'min_samples_leaf': 3} | RMSECV: 0.17319
 epoch    RMSEC    RMSEV   OOB_R2
    96 0.111461 0.143335 0.630291
    97 0.111591 0.143733 0.628119
    98 0.111502 0.143188 0.628753
    99 0.111668 0.143833 0.626683
   100 0.111676 0.143358 0.626166
                                                           0
Model                                                     RF
Preproc                                              snv_sg1
Params     {'ccp_alpha': 0.0, 'max_depth': 8, 'max_featur...
Split                  loaded from plsr_split_ks70_15_15.csv
RMSECV                                                0.1732
RMSE_oob                                              0.1771
RMSEV                                                 0.1434
R2_cal                                                0.8514
RMSE_cal                                              0.1117
RPD_cal         

In [3]:
# ------------------------------------------------------------
# 1D CNN on SpectroFood (RTX 4090, PyTorch + Optuna)
# ------------------------------------------------------------
import os, random
import numpy as np, pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.signal import savgol_filter
from sklearn.metrics import r2_score, mean_squared_error
import optuna

# ---- Config ----
CSV        = "SpectroFood_dataset.csv"
SPLIT_CSV  = "plsr_split_ks70_15_15.csv"
PREPROC    = "snv_sg1"
N_TRIALS   = 40
SEED       = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# ---- Reproducibility ----
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---- Data ----
df = pd.read_csv(CSV)
ID_COL, TARGET_COL = "Mush No", "Dry matter"
ids = df[ID_COL].values
y   = df[TARGET_COL].values.astype(np.float32)
X   = df.drop(columns=[ID_COL, TARGET_COL]).values.astype(np.float32)

def snv(A): return (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-9)
def preprocess(A, m):
    if m == "raw":     return A
    if m == "snv":     return snv(A)
    if m == "snv_sg1": return savgol_filter(snv(A), 11, 2, deriv=1, axis=1)
    raise ValueError(m)

Xp = preprocess(X, PREPROC).astype(np.float32)

# ---- Split ----
sp = pd.read_csv(SPLIT_CSV).set_index("Mush No").loc[ids].reset_index()
split = sp["split"].values
cal = np.where(split == "cal")[0]
val = np.where(split == "val")[0]
tst = np.where(split == "test")[0]
Xc, yc = Xp[cal], y[cal]
Xv, yv = Xp[val], y[val]
Xt, yt = Xp[tst], y[tst]

# ---- Standardize target using CAL only ----
y_mean, y_std = yc.mean(), yc.std()
yc_s = ((yc - y_mean) / y_std).astype(np.float32)
yv_s = ((yv - y_mean) / y_std).astype(np.float32)
yt_s = ((yt - y_mean) / y_std).astype(np.float32)
print(f"Cal {len(cal)} | Val {len(val)} | Test {len(tst)} | bands {Xp.shape[1]}")

# ---- Tensors ----
def to_loader(Xa, ya, bs, shuffle):
    # shape: (N, 1, bands)
    ds = TensorDataset(torch.from_numpy(Xa).unsqueeze(1),
                       torch.from_numpy(ya).unsqueeze(1))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == "cuda"))

# ---- Model ----
class CNN1D(nn.Module):
    def __init__(self, f1, f2, k1, k2, d1, d2, dd, du, l2):
        super().__init__()
        self.conv1 = nn.Conv1d(1,  f1, k1, padding=k1 // 2)
        self.conv2 = nn.Conv1d(f1, f2, k2, padding=k2 // 2)
        self.bn1   = nn.BatchNorm1d(f1)
        self.bn2   = nn.BatchNorm1d(f2)
        self.pool  = nn.MaxPool1d(2)
        self.drop1 = nn.Dropout(d1)
        self.drop2 = nn.Dropout(d2)
        self.dropd = nn.Dropout(dd)
        self.act   = nn.ReLU()
        # infer flatten size from a dummy forward
        with torch.no_grad():
            dummy = torch.zeros(1, 1, Xp.shape[1])
            out = self._features(dummy).shape[1]
        self.fc1 = nn.Linear(out, du)
        self.fc2 = nn.Linear(du, 1)
        self.l2  = l2

    def _features(self, x):
        x = self.act(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.drop1(x)
        x = self.act(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.drop2(x)
        return x.flatten(1)

    def forward(self, x):
        x = self._features(x)
        x = self.dropd(self.act(self.fc1(x)))
        return self.fc2(x)

# ---- Train one config, return val RMSE (original units) ----
def train_eval(f1, f2, k1, k2, d1, d2, dd, du, lr, l2, bs, epochs=100, patience=15):
    set_seed(SEED)
    tr_loader = to_loader(Xc, yc_s, bs, shuffle=True)
    va_loader = to_loader(Xv, yv_s, bs, shuffle=False)
    Xv_t = torch.from_numpy(Xv).unsqueeze(1).to(DEVICE)

    model = CNN1D(f1, f2, k1, k2, d1, d2, dd, du, l2).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)
    lossf = nn.MSELoss()

    best_val, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            preds = []
            for xb, _ in va_loader:
                preds.append(model(xb.to(DEVICE)).cpu())
            pv_s = torch.cat(preds).numpy().ravel()
        pv = pv_s * y_std + y_mean
        r  = float(np.sqrt(mean_squared_error(yv, pv)))
        if r < best_val:
            best_val = r; bad = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pt_s = model(torch.from_numpy(Xt).unsqueeze(1).to(DEVICE)).cpu().numpy().ravel()
    pt = pt_s * y_std + y_mean
    return best_val, model, pt

# ---- Optuna ----
def objective(trial):
    f1 = trial.suggest_categorical("filters_1", [16, 32, 64])
    f2 = trial.suggest_categorical("filters_2", [32, 64, 128, 256])
    k1 = trial.suggest_categorical("kernel_1", [3, 5, 7, 11])
    k2 = trial.suggest_categorical("kernel_2", [3, 5, 7])
    d1 = trial.suggest_float("dropout_1", 0.0, 0.5)
    d2 = trial.suggest_float("dropout_2", 0.0, 0.5)
    dd = trial.suggest_float("dropout_dense", 0.0, 0.5)
    du = trial.suggest_categorical("dense_units", [64, 128, 256])
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    l2 = trial.suggest_float("l2", 1e-6, 1e-2, log=True)
    bs = trial.suggest_categorical("batch_size", [16, 32, 64])
    r, _, _ = train_eval(f1, f2, k1, k2, d1, d2, dd, du, lr, l2, bs)
    return r

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nBest trial:")
print("  RMSEV:", round(study.best_value, 4))
print("  Params:", study.best_params)

# ---- Final model with best params (longer training) ----
bp = study.best_params
best_val, model, pt = train_eval(
    bp["filters_1"], bp["filters_2"], bp["kernel_1"], bp["kernel_2"],
    bp["dropout_1"], bp["dropout_2"], bp["dropout_dense"],
    bp["dense_units"], bp["lr"], bp["l2"], bp["batch_size"],
    epochs=300, patience=30,
)

model.eval()
with torch.no_grad():
    pc_s = model(torch.from_numpy(Xc).unsqueeze(1).to(DEVICE)).cpu().numpy().ravel()
    pv_s = model(torch.from_numpy(Xv).unsqueeze(1).to(DEVICE)).cpu().numpy().ravel()
pc = pc_s * y_std + y_mean
pv = pv_s * y_std + y_mean

def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def metrics(t, p):
    r = rmse(t, p)
    return dict(R2=r2_score(t, p), RMSE=r,
                RPD=np.std(t, ddof=1) / r,
                RPIQ=(np.percentile(t, 75) - np.percentile(t, 25)) / r,
                Bias=float(np.mean(p - t)))

res = {"Model": "CNN1D_PyTorch", "Preproc": PREPROC, "Device": str(DEVICE),
       "Params": str(bp), "RMSEV": best_val}
res.update({f"{k}_cal":  v for k, v in metrics(yc, pc).items()})
res.update({f"{k}_val":  v for k, v in metrics(yv, pv).items()})
res.update({f"{k}_test": v for k, v in metrics(yt, pt).items()})
print(pd.DataFrame([res]).round(4).T)

pd.DataFrame([res]).to_csv("cnn1d_pytorch_results.csv", index=False)
pd.DataFrame({"Mush No": ids[cal], "y_true": yc, "y_pred": pc}).to_csv("cnn1d_cal_pred.csv",  index=False)
pd.DataFrame({"Mush No": ids[val], "y_true": yv, "y_pred": pv}).to_csv("cnn1d_val_pred.csv",  index=False)
pd.DataFrame({"Mush No": ids[tst], "y_true": yt, "y_pred": pt}).to_csv("cnn1d_test_pred.csv", index=False)
print("\nSaved results + predictions.")

Device: cuda | NVIDIA GeForce RTX 4090
Cal 175 | Val 38 | Test 37 | bands 204


Best trial: 23. Best value: 0.0781596: 100%|██████████| 40/40 [01:03<00:00,  1.58s/it]



Best trial:
  RMSEV: 0.0782
  Params: {'filters_1': 32, 'filters_2': 32, 'kernel_1': 11, 'kernel_2': 7, 'dropout_1': 0.20767633290969467, 'dropout_2': 0.051511497869701356, 'dropout_dense': 0.10378316202451437, 'dense_units': 256, 'lr': 0.002111614074739296, 'l2': 0.00016894527180613415, 'batch_size': 16}
                                                           0
Model                                          CNN1D_PyTorch
Preproc                                              snv_sg1
Device                                                  cuda
Params     {'filters_1': 32, 'filters_2': 32, 'kernel_1':...
RMSEV                                                 0.0803
R2_cal                                                0.9359
RMSE_cal                                              0.0733
RPD_cal                                               3.9625
RPIQ_cal                                              2.8823
Bias_cal                                             -0.0272
R2_val               